# Phase 7 — Stage C: hard top-1 fine-tuning

Before Stage C, run a standalone dictionary-only coherence warmup from the Stage A dictionary. It must reach mu(D) < 0.20; the training cell then loads the Stage B model and overrides only its dictionary with the warmed checkpoint. Fine-tuning uses top-1 routing, 500 steps of logit jitter, initial-epoch task-loss normalization, adaptive load balance, and kappa=0.1. Stage C is complete only if the final checkpoint reaches mu(D) < 0.15 and per-task validation is non-decreasing in both PSNR and SSIM at every epoch. Outputs go to a new resumable Drive directory so the prior run is preserved. Before running, ensure the Colab clone's master branch contains these code changes; the notebook pulls master during setup. No prediction/intermediate images are written.

The acquisition cell prepares DIV2K, Flickr2K, CBSD68, SOTS, and Outdoor-Rain under Drive, reusing existing folders. DIV2K use is restricted to academic research. Flickr2K's EDSR archive and the CBSD68 community mirror (`https://github.com/clausmichele/CBSD68-dataset`) have public-source download attempts; if unavailable, supply a Drive archive/folder. SOTS is distributed from the [RESIDE dataset page](https://sites.google.com/view/reside-dehaze-datasets/reside-v0); access/download packaging varies. Outdoor-Rain does not identify a unique source in the roadmap, so select the intended corpus and provide an authorized direct URL or Drive archive/file ID. Urban100 acquisition from the EDSR archive is optional for the later benchmark task. Real paired rows can be supplied with `image_path,low_quality_path,task` CSV manifests.


## Data sources (updated 2026-11-03)

The original `cv.snu.ac.kr` EDSR host is dead, so the notebook now uses verified
alternatives: Flickr2K from the Hugging Face `yangtao9009` HR zip (10.85 GB,
downloaded only if you add `flickr2k` to `DATASETS`), CBSD68 from GitHub, DIV2K
from ETH Zurich, SOTS from the Kaggle `balraj98` mirror, Outdoor-Rain/deraining
from the Hugging Face `dronefreak/Rain13K` paired repository (Restormer's
standard deraining protocol), and Urban100 from the Hugging Face
`eugenesiow` HR tar.gz (129 MB), fetched for the Stage C benchmark.

Stage C additionally needs the finished Stage B checkpoint at
`/content/drive/MyDrive/SparseDict/phase7/stage_b/latest.pt`.
Run the cells in order: setup -> Kaggle credentials -> download -> manifests ->
training. The Kaggle cell prompts for your username + API key with masked
input (nothing is stored inside this notebook file); to avoid retyping each
session, save Colab Secrets named `KAGGLE_USERNAME` and `KAGGLE_API_KEY` and
the cell picks them up automatically. If Drive space is tight, keep
`DATASETS` trimmed to one new corpus per session - downloaded corpora are
reused and training resumes from `latest.pt`.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/SparseDict')
DATA_ROOT = Path('/content/drive/MyDrive/SparseDict/datasets')
RUN_DIR = Path('/content/drive/MyDrive/SparseDict/phase7/stage_c_coherence_retrain')
STAGE_B = Path('/content/drive/MyDrive/SparseDict/phase7/stage_b/latest.pt')
RUN_DIR.mkdir(parents=True, exist_ok=True)
if not REPO.exists():
    subprocess.run(['git', 'clone', '--branch', 'master', 'https://github.com/kalpeshjp89/SparseDict.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'master'], check=True)
sys.path.insert(0, str(REPO))
import torch
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime in Colab before training.')

In [ ]:
# @title 2) Kaggle credentials (needed for the SOTS download)
if (DATA_ROOT / 'SOTS').is_dir():
    print('SOTS already on Drive; skipping Kaggle credential setup.')
else:
    # Kaggle token setup - paste ONCE per runtime. On kaggle.com go to
    # Account -> API -> "Create New Token"; the downloaded kaggle.json contains
    # exactly two values: username and key. Input below is masked with getpass
    # and is never written into the notebook file.
    import json
    import os
    import subprocess
    import sys
    from pathlib import Path


    def _colab_secret(name: str) -> str:
        try:
            from google.colab import userdata

            return userdata.get(name)
        except Exception:
            return ""


    KAGGLE_USERNAME_INPUT = _colab_secret("KAGGLE_USERNAME")
    KAGGLE_API_KEY_INPUT = _colab_secret("KAGGLE_API_KEY")
    if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
        import getpass

        KAGGLE_USERNAME_INPUT = getpass.getpass("Kaggle username: ").strip()
        KAGGLE_API_KEY_INPUT = getpass.getpass("Kaggle API key: ").strip()

    KAGGLE_READY = False
    if not (KAGGLE_USERNAME_INPUT and KAGGLE_API_KEY_INPUT):
        print(
            "Kaggle credentials skipped. The `sots` kaggle: source below will "
            "fail until this cell is completed with username + key."
        )
    else:
        kaggle_dir = Path.home() / ".kaggle"
        kaggle_dir.mkdir(parents=True, exist_ok=True)
        token_path = kaggle_dir / "kaggle.json"
        token_path.write_text(
            json.dumps({"username": KAGGLE_USERNAME_INPUT, "key": KAGGLE_API_KEY_INPUT})
        )
        token_path.chmod(0o600)
        os.environ["KAGGLE_CONFIG_DIR"] = str(kaggle_dir)
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "kaggle"]
        ).check_returncode()
        result = subprocess.run(
            [
                "kaggle",
                "datasets",
                "files",
                "balraj98/synthetic-objective-testing-set-sots-reside",
            ],
            capture_output=True,
            text=True,
        )
        KAGGLE_READY = result.returncode == 0
        print(
            "kaggle auth check:",
            "OK" if KAGGLE_READY else (result.stderr or result.stdout).strip()[:300],
        )
        if KAGGLE_READY:
            print(
                "Kaggle credentials installed at ~/.kaggle/kaggle.json. If the "
                "check above reports a 403/permission error instead, accept the "
                "dataset once: open the dataset page on Kaggle while signed in "
                "and agree to its access rule, then rerun this cell."
            )

In [ ]:
# @title 3) Download the datasets (Drive), one corpus per session if space is tight
# Small -> large order. Already-downloaded folders on Drive are reused, so a
# rerun only fetches what is missing. Keep 'flickr2k' (10.85 GB HR zip) out
# of the list unless this session has room for it.
DATASETS = ['cbsd68', 'sots', 'outdoor-rain', 'div2k']
DOWNLOAD_PUBLIC_DATASETS = True
DOWNLOAD_URBAN100_FOR_BENCHMARK = True
# DIV2K is academic-research-only; explicitly accept its terms to download.
ACCEPT_DIV2K_TERMS = False
SOURCE_URLS = {
    # SOTS dehazing mirror (requires the Kaggle credentials cell to pass).
    'sots': 'kaggle:balraj98/synthetic-objective-testing-set-sots-reside',
    # Outdoor-Rain / deraining training pairs (Restormer's Rain13K protocol,
    # spread HF repository downloaded with huggingface_hub).
    'outdoor-rain': 'hf-dataset:dronefreak/Rain13K',
}
DRIVE_ARCHIVES = {}  # e.g. {'sots': '/content/drive/MyDrive/datasets/SOTS.zip'}
DRIVE_FILE_IDS = {}  # e.g. {'cbsd68': 'Google-Drive-file-id'}
if DOWNLOAD_URBAN100_FOR_BENCHMARK:
    DATASETS.append('urban100')  # 129 MB HR tar.gz from Hugging Face
command = [sys.executable, str(REPO / 'scripts' / 'prepare_phase7_datasets.py'),
           '--root', str(DATA_ROOT)]
for dataset in DATASETS:
    command.extend(['--dataset', dataset])
if DOWNLOAD_PUBLIC_DATASETS:
    command.append('--download-public')
if ACCEPT_DIV2K_TERMS:
    command.append('--accept-div2k-license')
for dataset, path in DRIVE_ARCHIVES.items():
    command.extend(['--drive-archive', f'{dataset}={path}'])
for dataset, file_id in DRIVE_FILE_IDS.items():
    command.extend(['--drive-file-id', f'{dataset}={file_id}'])
for dataset, url in SOURCE_URLS.items():
    command.extend(['--source-url', f'{dataset}={url}'])
import os
proc = subprocess.run(
    command, cwd=REPO, capture_output=True, text=True,
    env=dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True'),
)
print('returncode:', proc.returncode)
if proc.stdout:
    print('---- child stdout (tail) ----')
    print(proc.stdout[-6000:])
if proc.stderr:
    print('---- child stderr (tail) ----')
    print(proc.stderr[-6000:])

In [ ]:
from scripts.prepare_phase7_datasets import resolve_clean_image_dir
TRAIN_DATASETS = [name for name in DATASETS if name != 'urban100']
# Override a detected folder if a dataset uses a different clean/GT layout.
DATASET_PATH_OVERRIDES = {}  # e.g. {'sots': DATA_ROOT / 'SOTS' / 'outdoor' / 'GT'}
DATASET_DIRS = {
    name: Path(DATASET_PATH_OVERRIDES[name]) if name in DATASET_PATH_OVERRIDES
    else resolve_clean_image_dir(name, DATA_ROOT)
    for name in TRAIN_DATASETS
}
IMAGE_DIRS = list(DATASET_DIRS.values())
# The Kaggle SOTS mirror stores GT as separate indoor/outdoor halves. The
# resolver above returns only the first clean folder, so append any further
# gt/clear folders that exist next to it (skipping ones already selected).
_SOTS_ROOT = DATA_ROOT / 'SOTS'
_SOTS_IMAGE_SUFFIXES = {'.png', '.jpg', '.jpeg', '.bmp'}
if _SOTS_ROOT.is_dir():
    for sub in sorted(
        p for p in _SOTS_ROOT.rglob('*')
        if p.is_dir() and p.name.casefold() in {'gt', 'clear', 'target', 'ground_truth'}
        and any(f.suffix.lower() in _SOTS_IMAGE_SUFFIXES for f in p.rglob('*'))
    ):
        if sub not in IMAGE_DIRS:
            IMAGE_DIRS.append(sub)

if 'urban100' in DATASETS:
    URBAN100_DIR = resolve_clean_image_dir('urban100', DATA_ROOT)
    print('Urban100 benchmark images:', URBAN100_DIR)
print('Fine-tuning image folders:', DATASET_DIRS)
PAIRED_MANIFESTS = []
# Example: PAIRED_MANIFESTS.append(Path('/content/drive/MyDrive/datasets/paired_train.csv'))
if not all(path.is_dir() for path in IMAGE_DIRS):
    raise FileNotFoundError('Download DIV2K or configure IMAGE_DIRS first.')

TRAIN_MANIFEST = RUN_DIR / 'train.csv'
VALIDATION_MANIFEST = RUN_DIR / 'validation.csv'
TASKS = 'sr2,sr3,sr4,denoise15,denoise25,denoise50,derain,desnow,dehaze,lowlight,jpeg'
manifest_command = [
    sys.executable, str(REPO / 'scripts' / 'make_universal_manifest.py'),
    '--train-output', str(TRAIN_MANIFEST), '--validation-output', str(VALIDATION_MANIFEST),
    '--validation-fraction', '0.1', '--seed', '2027', '--tasks', TASKS
]
for directory in IMAGE_DIRS:
    manifest_command.extend(['--image-dir', str(directory)])
for paired_manifest in PAIRED_MANIFESTS:
    manifest_command.extend(['--paired-manifest', str(paired_manifest)])
subprocess.run(manifest_command, cwd=REPO, check=True)
if not STAGE_B.is_file():
    raise FileNotFoundError(f'Run Stage B first: {STAGE_B}')

In [ ]:
# Standalone coherence warmup (resume) + Stage C hard top-1 fine-tune. GPU required.
import time

RUN_DIR.mkdir(parents=True, exist_ok=True)
STAGE_A = Path('/content/drive/MyDrive/SparseDict/phase7/stage_a/pretrained_dictionaries.pth')
COHERENCE_CHECKPOINT = RUN_DIR / 'pretrained_dictionaries_incoherent.pth'
FAILURE_LOG = RUN_DIR / 'stage_c_failure.log'
ALLOC_ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

if not torch.cuda.is_available():
    raise SystemExit('GPU runtime required: warmup and Stage C both run on the T4.')
print('GPU:', torch.cuda.get_device_name(0))


def _warm_status():
    if not COHERENCE_CHECKPOINT.is_file():
        return None
    return torch.load(COHERENCE_CHECKPOINT, map_location='cpu', weights_only=False).get('coherence_warmup', {}).get('status')


if _warm_status() != 'target_met':
    # Resume from the partial local checkpoint (mu 0.2553 at 6k steps), else Stage A.
    init_ckpt = COHERENCE_CHECKPOINT if COHERENCE_CHECKPOINT.is_file() else STAGE_A
    print('Warmup resume from:', init_ckpt)
    warmup_cmd = [
        sys.executable, str(REPO / 'scripts' / 'warmup_coherence.py'),
        '--init-checkpoint', str(init_ckpt), '--output', str(COHERENCE_CHECKPOINT),
        '--steps', '30000', '--target', '0.20', '--learning-rate', '0.03',
        '--check-every', '50', '--device', 'cuda',
    ]
    print(' '.join(warmup_cmd))
    with open(FAILURE_LOG, 'a') as logf:
        logf.write('==== coherence warmup ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
        logf.flush()
        warmup = subprocess.Popen(warmup_cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ALLOC_ENV)
        for line in warmup.stdout:
            print(line, end='')
            logf.write(line)
            logf.flush()
        warmup.wait()
    print('warmup returncode:', warmup.returncode)
    if warmup.returncode != 0 or _warm_status() != 'target_met':
        raise RuntimeError('warmup did not reach mu < 0.20; Stage C not started. Log: %s' % FAILURE_LOG)
else:
    print('Warmup already target_met; skipping.')
print('Coherence checkpoint ready:', COHERENCE_CHECKPOINT)

# Stage C: batch is hard-coded 1; budget is --steps-per-epoch (reduced for stability after Kaggle OOM-style failures).
command = [
    sys.executable, str(REPO / 'experiments' / 'train_universal.py'),
    '--phase', 'fine_tune', '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST), '--init-checkpoint', str(STAGE_B),
    '--dictionary-checkpoint', str(COHERENCE_CHECKPOINT),
    '--output-dir', str(RUN_DIR), '--epochs', '20', '--warmup-epochs', '0',
    '--steps-per-epoch', '50', '--patch-size', '144',
    '--validation-samples-per-task', '8', '--coherence-weight', '0.001',
    '--coherence-target', '0.20',
    '--fine-tune-jitter-steps', '500', '--logit-jitter', '0.1',
    '--aux-weight', '0.01', '--collapsed-aux-weight', '0.05',
    '--collapsed-expert-threshold', '0.45', '--entropy-kappa', '0.1',
    '--checkpoint-every-steps', '25', '--device', 'cuda'
]
print(' '.join(command))
with open(FAILURE_LOG, 'a') as logf:
    logf.write('==== Stage C fine-tune ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
    logf.flush()
    proc = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ALLOC_ENV)
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
    proc.wait()
print('returncode:', proc.returncode)
if proc.returncode != 0:
    raise RuntimeError('Stage C failed (rc=%s). Full log: %s' % (proc.returncode, FAILURE_LOG))
print('Metrics and status:', RUN_DIR / 'metrics.jsonl', RUN_DIR / 'trainer_status.json')


In [ ]:
# Warmup decay follow-up (proven recipe, todo.md 7.1): lr 0.005, tau 0.005, stops at target.
import time
COHERENCE_CHECKPOINT = RUN_DIR / 'pretrained_dictionaries_incoherent.pth'
FAILURE_LOG = RUN_DIR / 'stage_c_failure.log'
ALLOC_ENV = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

def _warm_status():
    return torch.load(COHERENCE_CHECKPOINT, map_location='cpu', weights_only=False).get('coherence_warmup', {}).get('status')


for attempt in range(3):
    if _warm_status() == 'target_met':
        break
    cmd = [sys.executable, str(REPO / 'scripts' / 'warmup_coherence.py'),
           '--init-checkpoint', str(COHERENCE_CHECKPOINT), '--output', str(COHERENCE_CHECKPOINT),
           '--steps', '2000', '--target', '0.20', '--learning-rate', '0.005',
           '--max-temperature', '0.005', '--check-every', '10', '--device', 'cuda']
    print('decay pass', attempt)
    print(' '.join(cmd))
    with open(FAILURE_LOG, 'a') as logf:
        logf.write('==== warmup decay pass ' + str(attempt) + ' ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
        logf.flush()
        proc = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ALLOC_ENV)
        for line in proc.stdout:
            print(line, end='')
            logf.write(line)
            logf.flush()
        proc.wait()
    if proc.returncode != 0:
        raise RuntimeError('decay pass failed rc=%s' % proc.returncode)
status = _warm_status()
print('warmup status:', status)
if status != 'target_met':
    raise RuntimeError('decay did not reach mu < 0.20; Stage C not started. Log: %s' % FAILURE_LOG)
print('Coherence checkpoint ready:', COHERENCE_CHECKPOINT)

command = [
    sys.executable, str(REPO / 'experiments' / 'train_universal.py'),
    '--phase', 'fine_tune', '--train-manifest', str(TRAIN_MANIFEST),
    '--validation-manifest', str(VALIDATION_MANIFEST), '--init-checkpoint', str(STAGE_B),
    '--dictionary-checkpoint', str(COHERENCE_CHECKPOINT),
    '--output-dir', str(RUN_DIR), '--epochs', '20', '--warmup-epochs', '0',
    '--steps-per-epoch', '50', '--patch-size', '144',
    '--validation-samples-per-task', '8', '--coherence-weight', '0.001',
    '--coherence-target', '0.20',
    '--fine-tune-jitter-steps', '500', '--logit-jitter', '0.1',
    '--aux-weight', '0.01', '--collapsed-aux-weight', '0.05',
    '--collapsed-expert-threshold', '0.45', '--entropy-kappa', '0.1',
    '--checkpoint-every-steps', '25', '--device', 'cuda'
]
print(' '.join(command))
with open(FAILURE_LOG, 'a') as logf:
    logf.write('==== Stage C fine-tune ' + time.strftime('%Y-%m-%d %H:%M:%S') + ' ====' + chr(10))
    logf.flush()
    proc = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=ALLOC_ENV)
    for line in proc.stdout:
        print(line, end='')
        logf.write(line)
        logf.flush()
    proc.wait()
print('returncode:', proc.returncode)
if proc.returncode != 0:
    raise RuntimeError('Stage C failed (rc=%s). Full log: %s' % (proc.returncode, FAILURE_LOG))
print('Metrics and status:', RUN_DIR / 'metrics.jsonl', RUN_DIR / 'trainer_status.json')
